In [ ]:
import json
import requests
import openai

client = openai.OpenAI()
messages = []

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


In [ ]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다. 인기 영화를 추천하거나 보여줄 때 사용하세요.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "특정 영화 ID에 해당하는 영화의 상세 정보(제목, 줄거리, 평점 등)를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "정보를 가져올 영화의 고유 ID",
                    },
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "특정 영화 ID에 해당하는 영화의 출연진(배우) 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "출연진 정보를 가져올 영화의 고유 ID",
                    },
                },
                "required": ["id"],
            },
        },
    },
]


In [ ]:
def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    choice = response.choices[0]

    # 모델이 도구 호출을 요청한 경우
    if choice.finish_reason == "tool_calls":
        tool_calls = choice.message.tool_calls
        messages.append(choice.message)  # assistant 메시지(tool_calls 포함) 추가

        for tool_call in tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            print(f"[도구 호출] {fn_name}({fn_args})")

            fn_result = FUNCTION_MAP[fn_name](**fn_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(fn_result, ensure_ascii=False),
            })

        # 도구 결과를 바탕으로 최종 응답 생성
        call_ai()
    else:
        message = choice.message.content
        messages.append({"role": "assistant", "content": message})
        print(f"AI: {message}")


In [ ]:
while True:
    user_input = input("메시지를 입력하세요 (종료: q): ")
    if user_input.lower() == "q":
        break
    print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai()
